# Data Loading and Preparation

In [ ]:
# Basic imports
import pandas as pd
import numpy as np

In [ ]:
# Load raw UNSW-NB15 datasets and feature names
from pathlib import Path

feat_path = Path("../data/raw/UNSW-NB15_features.csv")
data_path1 = Path("../data/raw/UNSW-NB15_1.csv")
data_path2 = Path("../data/raw/UNSW-NB15_2.csv")

# Read the features file
feat_df = pd.read_csv(feat_path, encoding="ISO-8859-1")

In [ ]:
# Combine UNSW-NB15 CSVs into a single DataFrame
col_names = feat_df["Name"].tolist()
df1 = pd.read_csv(data_path1, names=col_names, low_memory=False)
df2 = pd.read_csv(data_path2, names=col_names, low_memory=False)
df_raw = pd.concat([df1, df2], ignore_index=True) # Ensures unique indices across files

In [ ]:
# Prepare dataset for feature engineering
df_features = df_raw.copy() # Use the full dataset

# Byte/Volume-Based Features

### Log-Scale Destination Bytes
A `log_dbytes` feature is created by applying `log1p` to the `dbytes` column.
This reduces skew in byte counts and compresses large values, making the feature more suitable for modelling.

### Log-Scale Minimum Flow Bytes
A `log_min_flow_bytes` feature is created by applying `log1p` to the minimum of the `sbytes` and `dbytes` columns.
- Captures intensity of attack asymmetry/imbalance, highlighting cases where one of source or destination bytes is much smaller than the other.
- Log-scaling reduces skew and compresses large values, making the feature more suitable for modelling.

### Log-Scale Byte Product
A `log_byte_prod` feature is created by applying `log1p` to the product of the `sbytes` and `dbytes` columns.
- Emphasises asymmetric attack flows while downweighting/normalising balanced normal flows.
- Log-scaling reduces skew and compresses large values, making the feature more suitable for modelling.

In [ ]:
# Log-scale destination bytes
df_features["log_dbytes"] = np.log1p(df_features["dbytes"])

# Log-scale minimum flow bytes
df_features["log_min_flow_bytes"] = np.log1p(np.minimum(df_features["sbytes"], df_features["dbytes"]))

# Log-scale byte product
df_features["log_byte_prod"] = np.log1p(df_features["sbytes"]*df_features["dbytes"])

# Time-Based Features

### Load Skew
A `load_skew` feature is created by applying `log1p` to the ratio of the log-scaled `Dload` and `Sload` columns.
- Captures directional imbalance between destination and source loads, highlighting flows where one side dominates.
- Log-scaling reduces skew and compresses large values, making the feature more suitable for modelling.

In [ ]:
# Log-scale load ratio
df_features["load_skew"] = np.log1p(df_features["Dload"])/(np.log1p(df_features["Sload"]) + 1e-6)
df_features["load_skew"] = df_features["load_skew"].clip(upper=df_features["load_skew"].quantile(0.999)) # Clip extreme values

# Packet-Level Features

### Log-Scale Destination Mean Packet Size
A `log_dmeansz` feature is created by applying `log1p` to the `dmeansz` column.
This reduces skew in byte counts and compresses large values, making the feature more suitable for modelling.

### Mean Packet Size Ratio
A `mean_pkt_sz_ratio` feature is created by taking the ratio of the log-scaled `dmeansz` and `smeansz` columns.
- Captures directional imbalance between destination and source mean packet sizes, highlighting flows where one side dominates.
- Log-scaling reduces skew and compresses large values, making the feature more suitable for modelling.

In [ ]:
# Log-scale destination mean packet size
df_features["log_dmeansz"] = np.log1p(df_features["dmeansz"])

# Mean packet size ratio
df_features["mean_pkt_sz_ratio"] = np.log1p(df_features["dmeansz"])/(np.log1p(df_features["smeansz"]) + 1e-6)
df_features["mean_pkt_sz_ratio"] = df_features["mean_pkt_sz_ratio"].clip(upper=df_features["mean_pkt_sz_ratio"].quantile(0.999))

# TTL Features

### TTL Difference

A `ttl_diff` feature is created by taking the absolute difference of the `sttl` and `dttl` columns. This will highlight asymmetric TTls which can signal unusual routing, spoofing or other anomalous behaviour.

In [ ]:
# TTL difference
df_features["ttl_diff"] = abs(df_features["sttl"] - df_features["dttl"])

# Export Final Feature Set

In [ ]:
# Save df_features to CSV for modelling
df_features.to_csv("../data/processed/df_features.csv", index=False)